In [ ]:
prefix_path = '../..'
import sys
sys.path.append(prefix_path)

import os
import gc
from collections import defaultdict, Counter
import re
from tqdm import tqdm

import torch
import torch.nn.functional as F
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from sklearn.model_selection import train_test_split

from egh_vlm.hallu_dataset import load_hallu_dataset
from egh_vlm.utils import Qwen3ModelBundle, get_pred, load_phd_dataset

## Load Dataset and Features

In [ ]:
DATASET_PATH = f'{prefix_path}/data/phd/phd_base_qwen3.json'
IMG_FOLDER_PATH = f'{prefix_path}/data/phd/images'
FEATURES_PATH = f'{prefix_path}/data/phd/features/features_egh_full_context_phd_base_qwen3.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
TRAIN_RATIO = 0.7

In [ ]:
dataset = load_phd_dataset(
    dataset_path=DATASET_PATH,
    img_folder_path=IMG_FOLDER_PATH)

features = load_hallu_dataset(FEATURES_PATH) if os.path.isfile(FEATURES_PATH) else None
print(f'Length of processed features: {len(features) if features is not None else 0}')

In [ ]:
dataset[0]

## Build Contrastive Pairs (truthful, hallucinated) per couple

In [ ]:
# (intentionally empty — imports moved to the top cell)

In [ ]:
if features is None:
    raise ValueError("Features not found. Check FEATURES_PATH.")

feature_by_id = {
    fid: (emb, grad, label)
    for fid, emb, grad, label in zip(
        features.ids, features.embs, features.grads, features.labels
    )
}

combined_dataset = []
for item in dataset:
    feat = feature_by_id.get(item["id"])
    if feat is None:
        continue
    emb, grad, f_label = feat
    combined_dataset.append({**item, "emb": emb, "grad": grad, "feature_label": f_label})
print(f"Combined dataset: {len(combined_dataset)} / {len(dataset)}")

train_dataset, test_dataset = train_test_split(
    combined_dataset,
    train_size=TRAIN_RATIO,
    stratify=[it["label"] for it in combined_dataset],
    random_state=42,
)


def build_paired_dataset(ds):
    """Pair truthful (label==0) with hallucinated (label==1) within each couple_id."""
    by_couple = defaultdict(list)
    for it in ds:
        by_couple[it['couple_id']].append(it)

    pairs, skipped = [], {'no_truthful': 0, 'no_hallu': 0}
    for cid, items in by_couple.items():
        truthful = [i for i in items if i['label'] == 0]
        hallu    = [i for i in items if i['label'] == 1]
        if not truthful:
            skipped['no_truthful'] += 1; continue
        if not hallu:
            skipped['no_hallu'] += 1; continue
        for h in hallu:
            t = next(
                (x for x in truthful
                 if x['task'] == h['task'] and x['subject'] == h['subject']
                 and x['hitem'] == h['hitem']),
                truthful[0]
            )
            pairs.append({'couple_id': cid, 'task': h['task'], 'truthful': t, 'hallucinated': h})
    print(f"Paired couples: {len(pairs)} | skipped: {skipped}")
    return pairs


paired_dataset = build_paired_dataset(train_dataset)

In [ ]:
task_counts = Counter(p['task'] for p in paired_dataset)
total = sum(task_counts.values())
print(f"Total pairs: {total}")
for task, c in task_counts.most_common():
    print(f"  {task:12s} {c:5d} ({100*c/total:.1f}%)")

In [ ]:
## Build Task-Conditioned Steering Vectors
#
# NOTE on the features stored in extract_features.py:
#   emb  = c_emb - a_emb  (context-induced shift of the answer hidden state)
#   grad = d(KL)/d a_emb
#
# IMPORTANT: In the PhD dataset, hallucinated answers are heavily biased toward
# "yes" (false affirmatives on leading questions). A naive mean(T)-mean(H)
# contrast therefore captures mostly the "no - yes" token-polarity axis rather
# than the truthfulness axis. Set `balance_by_gt=True` to stratify the contrast
# by the hallucinated member's `question_gt`: we compute a contrast for gt=0
# (hallu_yes -> truthful_no) and for gt=1 (hallu_no -> truthful_yes), then sum
# them. The polarity components cancel; the truthfulness component adds.


def pool_features(emb, grad, mode='grad_weighted'):
    if emb.numel() == 0:
        return None
    if mode == 'mean':
        return emb.mean(0)
    if mode == 'grad_weighted':
        w = grad.abs().sum(-1)
        w = w / (w.sum() + 1e-8)
        return (emb * w.unsqueeze(-1)).sum(0)
    if mode == 'grad_dir':
        return grad.mean(0)
    if mode == 'last_token':
        return emb[-1]
    raise ValueError(mode)


def compute_task_steering_vectors(
    paired_dataset,
    hallu_features,
    pool_mode='grad_weighted',
    balance_by_gt=True,
    normalize=True,
    verbose=True,
):
    feat_by_id = {
        i: (e, g) for i, e, g in
        zip(hallu_features.ids, hallu_features.embs, hallu_features.grads)
    }

    # bucket[task][gt] = {'truth': [...], 'hallu': [...]}
    bucket = defaultdict(lambda: {0: {'truth': [], 'hallu': []},
                                  1: {'truth': [], 'hallu': []}})
    for p in paired_dataset:
        tid, hid = p['truthful']['id'], p['hallucinated']['id']
        if tid not in feat_by_id or hid not in feat_by_id:
            continue
        vt = pool_features(*feat_by_id[tid], mode=pool_mode)
        vh = pool_features(*feat_by_id[hid], mode=pool_mode)
        if vt is None or vh is None:
            continue
        gt = int(p['hallucinated']['question_gt'])
        if gt not in (0, 1):
            continue
        bucket[p['task']][gt]['truth'].append(vt)
        bucket[p['task']][gt]['hallu'].append(vh)

    steering = {}
    for task, per_gt in bucket.items():
        parts, counts = [], {}
        for gt, b in per_gt.items():
            if not b['truth'] or not b['hallu']:
                counts[gt] = 0
                continue
            T = torch.stack(b['truth']).float()
            H = torch.stack(b['hallu']).float()
            parts.append(T.mean(0) - H.mean(0))      # contrast per gt group
            counts[gt] = T.shape[0]

        if not parts:
            continue
        if balance_by_gt and len(parts) == 2:
            v = parts[0] + parts[1]                   # cancels yes/no axis
        else:
            # fallback: weighted sum (equivalent to unbalanced mean(T)-mean(H))
            v = sum(parts)
        raw_norm = v.norm().item()
        if normalize:
            v = v / (v.norm() + 1e-8)

        steering[task] = {
            'vector': v,
            'n_gt0': counts.get(0, 0),
            'n_gt1': counts.get(1, 0),
            'raw_norm': raw_norm,
        }
        if verbose:
            print(f"[{task:12s}] gt0={counts.get(0,0):4d}  gt1={counts.get(1,0):4d}  "
                  f"raw_norm={raw_norm:.3f}")
    return steering


task_steering = compute_task_steering_vectors(
    paired_dataset, features,
    pool_mode='grad_weighted', balance_by_gt=True,
)

In [ ]:
## Steering Hook
#
# Fixes vs prior version:
#  - call_count + last_delta_norm + last_hidden_norm so you can VERIFY the hook fires
#    and SEE the perturbation/activation ratio.
#  - IMPORTANT: we steer the LAST position on EVERY forward pass (prefill + decode).
#    The first generated token (the decisive "yes"/"no") is produced by the PREFILL
#    pass, where seq_len == prompt_len, not 1. Gating on seq_len==1 skips exactly
#    that token and leaves the yes/no decision unchanged -- which is what we were
#    seeing earlier (explanation text changed but yes/no didn't).
#    Because we only write to hs[:, -1, :], earlier image/prompt tokens are never
#    modified, so there's no encoding corruption.
#  - `alpha_relative=True` rescales alpha so the added perturbation has norm
#    `alpha * ||h_last||`. With this, alpha is interpretable as a fraction of the
#    activation magnitude (try 0.05 .. 0.5).


def _find_decoder_layers(model):
    candidates = [
        lambda m: m.model.language_model.layers,
        lambda m: m.language_model.model.layers,
        lambda m: m.language_model.layers,
        lambda m: m.model.layers,
        lambda m: m.transformer.h,
    ]
    for fn in candidates:
        try:
            layers = fn(model)
            if hasattr(layers, '__len__') and len(layers) > 0:
                return layers
        except AttributeError:
            continue
    raise AttributeError("Could not locate decoder layers.")


def _build_messages(item, prompt_text=None):
    question = item['question']
    if prompt_text:
        question = prompt_text.format(question=question)
    content = []
    if item.get('image_path'):
        content.append({'type': 'image', 'image': item['image_path']})
    content.append({'type': 'text', 'text': question})
    return [{'role': 'user', 'content': content}]


class SteeringHook:
    def __init__(self, model, layer_idx, vector, alpha=1.0,
                 only_new_tokens=False, alpha_relative=True):
        layers = _find_decoder_layers(model)
        if layer_idx < 0:
            idx = len(layers) + layer_idx
        elif layer_idx > 0:
            idx = layer_idx - 1   # hidden_states[L] == output of layers[L-1]
        else:
            idx = 0
        self.idx = idx
        self.n_layers = len(layers)

        p = next(model.parameters())
        self.vector = vector.to(device=p.device, dtype=p.dtype)
        self.alpha = alpha
        self.only_new_tokens = only_new_tokens
        self.alpha_relative = alpha_relative

        # diagnostics
        self.call_count = 0
        self.applied_count = 0
        self.last_delta_norm = 0.0
        self.last_hidden_norm = 0.0

        self.handle = layers[idx].register_forward_hook(self._hook)

    def _hook(self, module, inputs, output):
        hs = output[0] if isinstance(output, tuple) else output
        self.call_count += 1

        # If requested, skip prefill (seq_len > 1). DEFAULT IS FALSE because the
        # first generated token (yes/no) is produced during prefill.
        if self.only_new_tokens and hs.shape[1] != 1:
            return output

        last = hs[:, -1, :]                              # [B, H]
        h_norm = last.float().norm(dim=-1).mean().item()
        v_norm = self.vector.float().norm().item() + 1e-8

        if self.alpha_relative:
            scale = self.alpha * h_norm / v_norm
        else:
            scale = self.alpha

        delta = scale * self.vector                      # [H]
        hs = hs.clone()
        hs[:, -1, :] = last + delta.to(last.dtype)

        self.applied_count += 1
        self.last_delta_norm = float(delta.float().norm().item())
        self.last_hidden_norm = h_norm

        return (hs,) + output[1:] if isinstance(output, tuple) else hs

    def remove(self):
        self.handle.remove()


@torch.no_grad()
def generate_baseline(item, model_bundle, prompt_text=None, max_new_tokens=64, **gk):
    model, processor, device = model_bundle.model, model_bundle.processor, model_bundle.device
    messages = _build_messages(item, prompt_text)
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_dict=True, return_tensors='pt',
    ).to(device)
    gk.setdefault('max_new_tokens', max_new_tokens)
    out = model.generate(**inputs, **gk)
    return processor.batch_decode(
        out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
    )[0]


@torch.no_grad()
def generate_with_steering(
    item, model_bundle, task_steering, prompt_text=None,
    alpha=0.15, layer_idx=-8, max_new_tokens=64,
    only_new_tokens=False, alpha_relative=True, debug=False, **gk,
):
    model, processor, device = model_bundle.model, model_bundle.processor, model_bundle.device
    v = task_steering[item['task']]['vector']
    hook = SteeringHook(
        model, layer_idx, v, alpha=alpha,
        only_new_tokens=only_new_tokens, alpha_relative=alpha_relative,
    )
    try:
        messages = _build_messages(item, prompt_text)
        inputs = processor.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_dict=True, return_tensors='pt',
        ).to(device)
        gk.setdefault('max_new_tokens', max_new_tokens)
        out = model.generate(**inputs, **gk)
        text = processor.batch_decode(
            out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True,
        )[0]
        if debug:
            print(f"  hook: layer={hook.idx}/{hook.n_layers}  "
                  f"calls={hook.call_count} applied={hook.applied_count}  "
                  f"||delta||={hook.last_delta_norm:.2f}  "
                  f"||h||={hook.last_hidden_norm:.2f}")
        return text
    finally:
        hook.remove()

In [ ]:
## Evaluation Loop


@torch.no_grad()
def evaluate_hallucinated(
    dataset, model_bundle, task_steering=None,
    alpha=0.15, layer_idx=-8, only_new_tokens=False, alpha_relative=True,
    max_new_tokens=64, prompt_text=None, limit=None, debug_first=0,
):
    hallu_items = [it for it in dataset if it['label'] == 1]
    if limit:
        hallu_items = hallu_items[:limit]

    stats = {'total': len(hallu_items), 'fixed': 0, 'still_hallu': 0,
             'unparseable': 0, 'per_task': {}}
    records = []
    for i, item in enumerate(tqdm(hallu_items, desc='Evaluating')):
        gt = item['question_gt']
        if task_steering is not None and item['task'] in task_steering:
            response = generate_with_steering(
                item, model_bundle, task_steering,
                prompt_text=prompt_text, alpha=alpha, layer_idx=layer_idx,
                only_new_tokens=only_new_tokens, alpha_relative=alpha_relative,
                max_new_tokens=max_new_tokens, do_sample=False,
                debug=(i < debug_first),
            )
        else:
            response = generate_baseline(
                item, model_bundle, prompt_text=prompt_text,
                max_new_tokens=max_new_tokens, do_sample=False,
            )
        pred = get_pred(response)
        outcome = 'unparseable' if pred == 0.5 else ('fixed' if pred == gt else 'still_hallu')
        stats[outcome] += 1
        ts = stats['per_task'].setdefault(item['task'],
            {'total': 0, 'fixed': 0, 'still_hallu': 0, 'unparseable': 0})
        ts['total'] += 1; ts[outcome] += 1
        records.append({
            'id': item['id'], 'couple_id': item['couple_id'], 'task': item['task'],
            'question': item['question'], 'question_gt': gt,
            'old_answer': item['answer'], 'new_response': response,
            'pred': pred, 'outcome': outcome,
        })

    n = max(stats['total'], 1)
    print(f"\n=== Overall (n={stats['total']}) ===")
    print(f"  fixed:       {stats['fixed']:5d}  ({stats['fixed']/n:.1%})")
    print(f"  still hallu: {stats['still_hallu']:5d}  ({stats['still_hallu']/n:.1%})")
    print(f"  unparseable: {stats['unparseable']:5d}  ({stats['unparseable']/n:.1%})")
    print("=== Per task ===")
    for t, s in sorted(stats['per_task'].items()):
        m = max(s['total'], 1)
        print(f"  {t:12s} n={s['total']:4d} | fixed={s['fixed']/m:6.1%} | "
              f"still={s['still_hallu']/m:6.1%} | unparse={s['unparseable']/m:6.1%}")
    return stats, records

In [ ]:
model = Qwen3VLForConditionalGeneration.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    dtype=torch.float16,
    device_map=DEVICE
)
print("config.dtype:", model.config.dtype)
processor = AutoProcessor.from_pretrained(
    'Qwen/Qwen3-VL-2B-Instruct',
    max_pixels=1024 * 1024)
model_bundle = Qwen3ModelBundle(model, processor, DEVICE)

In [ ]:
layers = _find_decoder_layers(model_bundle.model)
print(type(layers).__name__, len(layers))

In [ ]:
PROMPT_PATH = f'{prefix_path}/data/phd/prompts/generate.txt'
with open(PROMPT_PATH, 'r', encoding='utf-8') as f:
    PROMPT_TEXT = f.read()

# ----- baseline -----
base_stats, base_records = evaluate_hallucinated(
    test_dataset,
    model_bundle,
    task_steering=None,
    prompt_text=PROMPT_TEXT,
    max_new_tokens=64,
    limit=20,
)

In [ ]:
base_records

In [ ]:
# # ----- steered: relative alpha (alpha is a fraction of ||h||) -----
# # Try alpha in {0.05, 0.1, 0.2, 0.3, 0.5} and layer_idx in {-14, -10, -8, -6}.
# # If results == baseline, the hook is not moving the distribution: increase alpha.
# # If outputs become gibberish, decrease alpha or move layer earlier.
# steer_stats, steer_records = evaluate_hallucinated(
#     test_dataset,
#     model_bundle,
#     task_steering=task_steering,
#     alpha=0.15,
#     layer_idx=-8,                # ~ 70% depth on a 28-layer model
#     only_new_tokens=True,
#     alpha_relative=True,
#     prompt_text=PROMPT_TEXT,
#     max_new_tokens=64,
#     limit=20,
#     debug_first=3,
# )

In [ ]:
# steer_records

## Cheap diagnostics: polarity-balanced eval + yes/no orthogonalization
# (1) Balance the test split 50/50 on `question_gt` so the yes/no shortcut
#     cannot carry the score.
# (2) Project the yes/no token-embedding axis out of each steering vector,
#     so what remains is (at most) polarity-independent truthfulness signal.
# If steering still does nothing after (2) on the balanced split from (1),
# the features carry no truthfulness content beyond the yes/no shortcut.

In [ ]:
import random


def build_balanced_eval_split(ds, seed=42, max_per_gt=None):
    """Return the hallucinated (label==1) subset with 50/50 split on question_gt.
    If max_per_gt is given, cap each polarity at that size."""
    hallu = [it for it in ds if it['label'] == 1]
    gt0 = [it for it in hallu if int(it['question_gt']) == 0]
    gt1 = [it for it in hallu if int(it['question_gt']) == 1]
    k = min(len(gt0), len(gt1))
    if max_per_gt is not None:
        k = min(k, max_per_gt)
    rng = random.Random(seed)
    gt0 = rng.sample(gt0, k)
    gt1 = rng.sample(gt1, k)
    balanced = gt0 + gt1
    rng.shuffle(balanced)
    print(f"Balanced split: {len(balanced)} ({k} gt=0 + {k} gt=1). "
          f"Original hallu: {len(hallu)} "
          f"({sum(1 for x in hallu if x['question_gt']==0)} gt=0 / "
          f"{sum(1 for x in hallu if x['question_gt']==1)} gt=1).")
    return balanced


balanced_test = build_balanced_eval_split(test_dataset, max_per_gt=40)

In [ ]:
def get_yes_no_axis(model_bundle, source='unembed',
                    variants=((' yes', ' no'), ('yes', 'no'),
                              ('Yes', 'No'), (' Yes', ' No'))):
    """Return a unit vector pointing from 'no' to 'yes' in:
        source='input'   -> input embedding space  (W_E[yes] - W_E[no])
        source='unembed' -> LM head space          (W_U[yes] - W_U[no])
    For Qwen3 these are NOT tied, so 'unembed' is the right choice for
    orthogonalizing a steering vector applied to the residual stream.
    """
    tok = model_bundle.processor.tokenizer
    model = model_bundle.model
    if source == 'input':
        W = model.get_input_embeddings().weight.detach().float().cpu()
    elif source == 'unembed':
        # Find the LM head across common Qwen/HF layouts.
        for getter in (lambda m: m.lm_head.weight,
                       lambda m: m.language_model.lm_head.weight,
                       lambda m: m.model.lm_head.weight):
            try:
                W = getter(model).detach().float().cpu()
                break
            except AttributeError:
                continue
        else:
            raise AttributeError("Could not locate lm_head.")
    else:
        raise ValueError(source)

    diffs = []
    for y, n in variants:
        y_ids = tok.encode(y, add_special_tokens=False)
        n_ids = tok.encode(n, add_special_tokens=False)
        if len(y_ids) != 1 or len(n_ids) != 1:
            continue
        diffs.append(W[y_ids[0]] - W[n_ids[0]])
    if not diffs:
        raise RuntimeError("Could not find single-token 'yes'/'no' variants.")
    pol = torch.stack(diffs).mean(0)
    pol = pol / (pol.norm() + 1e-8)
    print(f"[{source}] yes/no axis from {len(diffs)} variants; dim={pol.shape[0]}")
    return pol


def orthogonalize_task_steering(task_steering, polarity_axis, normalize=True):
    out = {}
    for task, info in task_steering.items():
        v = info['vector'].float()
        proj = float(v @ polarity_axis)
        v_perp = v - proj * polarity_axis
        frac_removed = (proj ** 2) / (v.norm() ** 2 + 1e-12)
        new_norm = v_perp.norm().item()
        if normalize:
            v_perp = v_perp / (new_norm + 1e-8)
        out[task] = {**info, 'vector': v_perp,
                     'polarity_projection': proj,
                     'polarity_energy_fraction': float(frac_removed),
                     'residual_norm': new_norm}
        print(f"[{task:12s}] proj={proj:+.3f}  energy={frac_removed:.1%}  "
              f"residual_norm={new_norm:.3f}")
    return out


# 1) Input-embedding axis (what we tried first; expect near-zero energy).
print("\n--- input-embedding axis ---")
pol_input = get_yes_no_axis(model_bundle, source='input')

# 2) Unembedding axis (what actually controls the logit).
print("\n--- unembedding (lm_head) axis ---")
pol_unembed = get_yes_no_axis(model_bundle, source='unembed')

# Build balanced contrast.
task_steering_bal = compute_task_steering_vectors(
    paired_dataset, features,
    pool_mode='grad_weighted', balance_by_gt=True, normalize=True, verbose=False,
)

print("\n--- orthogonalize against INPUT axis ---")
task_steering_ortho_in = orthogonalize_task_steering(task_steering_bal, pol_input)

print("\n--- orthogonalize against UNEMBED axis ---")
task_steering_ortho_un = orthogonalize_task_steering(task_steering_bal, pol_unembed)

In [ ]:
print("\n##### Baseline on balanced split #####")
bal_base_stats, bal_base_recs = evaluate_hallucinated(
    balanced_test, model_bundle, task_steering=None,
    prompt_text=PROMPT_TEXT, max_new_tokens=48,
)

print("\n##### Steering (balanced contrast, NOT orthogonalized) #####")
bal_bal_stats, bal_bal_recs = evaluate_hallucinated(
    balanced_test, model_bundle, task_steering=task_steering_bal,
    alpha=0.15, layer_idx=-10, prompt_text=PROMPT_TEXT, max_new_tokens=48,
)

print("\n##### Steering (orthogonalized vs INPUT embed yes/no) #####")
bal_in_stats, bal_in_recs = evaluate_hallucinated(
    balanced_test, model_bundle, task_steering=task_steering_ortho_in,
    alpha=0.15, layer_idx=-10, prompt_text=PROMPT_TEXT, max_new_tokens=48,
)

print("\n##### Steering (orthogonalized vs UNEMBED yes/no) #####")
bal_un_stats, bal_un_recs = evaluate_hallucinated(
    balanced_test, model_bundle, task_steering=task_steering_ortho_un,
    alpha=0.15, layer_idx=-10, prompt_text=PROMPT_TEXT, max_new_tokens=48,
)


def _per_gt(recs):
    g = {0: [0, 0], 1: [0, 0]}
    for r in recs:
        gt = int(r['question_gt'])
        if gt in g:
            g[gt][1] += 1
            if r['outcome'] == 'fixed':
                g[gt][0] += 1
    return {k: (v[0] / v[1] if v[1] else 0.0, v[1]) for k, v in g.items()}


def _print_row(tag, stats, recs):
    g = _per_gt(recs)
    n = max(stats['total'], 1)
    print(f"  {tag:28s} fixed={stats['fixed']/n:5.1%} "
          f"| gt=0 {g[0][0]:5.1%} (n={g[0][1]}) "
          f"| gt=1 {g[1][0]:5.1%} (n={g[1][1]})")


print("\n=== Summary on balanced split ===")
_print_row("baseline",               bal_base_stats, bal_base_recs)
_print_row("bal contrast (raw)",     bal_bal_stats,  bal_bal_recs)
_print_row("bal contrast ⊥ input",   bal_in_stats,   bal_in_recs)
_print_row("bal contrast ⊥ unembed", bal_un_stats,   bal_un_recs)

In [ ]:
# Final cheap rule-out: sweep (layer, alpha) on the orthogonalized balanced vector.
# If no (layer, alpha) lifts BOTH gt=0 and gt=1 above baseline by more than
# ~5pp, the features carry no usable polarity-independent steering signal.

LAYER_GRID = [-6, -3]
ALPHA_GRID = [0.15, 0.3, 0.6, 1.0]

print(f"Baseline: gt0={ _per_gt(bal_base_recs)[0][0]:.1%}  "
      f"gt1={ _per_gt(bal_base_recs)[1][0]:.1%}\n")

ortho_grid = {}
for L in LAYER_GRID:
    for a in ALPHA_GRID:
        stats, recs = evaluate_hallucinated(
            balanced_test, model_bundle,
            task_steering=task_steering_ortho_un,
            alpha=a, layer_idx=L,
            prompt_text=PROMPT_TEXT, max_new_tokens=48,
        )
        g = _per_gt(recs)
        ortho_grid[(L, a)] = (stats['fixed']/max(stats['total'],1), g[0][0], g[1][0])
        print(f"  L={L:+3d} α={a:<4} | fixed={ortho_grid[(L,a)][0]:5.1%} "
              f"| gt0={g[0][0]:5.1%} | gt1={g[1][0]:5.1%}")

base = _per_gt(bal_base_recs)
print("\n>>> rows where BOTH gt=0 and gt=1 strictly exceed baseline:")
hits = [(k, v) for k, v in ortho_grid.items() if v[1] > base[0][0] and v[2] > base[1][0]]
if not hits:
    print("  (none) -- features lack polarity-independent steering signal at any tested layer/alpha.")
else:
    for k, v in sorted(hits, key=lambda x: -(x[1][1]+x[1][2])):
        print(f"  L={k[0]:+3d} α={k[1]:<4} -> gt0={v[1]:.1%}  gt1={v[2]:.1%}")

In [ ]:
def quick_diff(test_items, model_bundle, task_steering, alpha, layer_idx,
               prompt_text=None, n=10, **gk):
    sample = [it for it in test_items if it['label'] == 1][:n]
    n_changed = 0
    for it in sample:
        b = generate_baseline(it, model_bundle, prompt_text=prompt_text,
                              max_new_tokens=32, do_sample=False)
        s = generate_with_steering(it, model_bundle, task_steering,
                                   prompt_text=prompt_text, alpha=alpha,
                                   layer_idx=layer_idx, max_new_tokens=32,
                                   do_sample=False, debug=False, **gk)
        if b.strip() != s.strip():
            n_changed += 1
    print(f"alpha={alpha} layer={layer_idx} -> {n_changed}/{n} outputs changed")
    return n_changed

for alpha in [0.05, 0.15, 0.3, 0.6, 1.0]:
    for layer_idx in [-14, -10, -8, -6, -3]:
        quick_diff(test_dataset, model_bundle, task_steering,
                   alpha=alpha, layer_idx=layer_idx,
                   prompt_text=PROMPT_TEXT, n=8)

## Sign / pooling sweep
Once you've found an `(alpha, layer)` that meaningfully changes outputs,
sweep pooling mode and sign on a small subset.

In [ ]:
def _polarity_breakdown(records):
    by_gt = {0: [0, 0], 1: [0, 0]}
    for r in records:
        gt = r['question_gt']
        if gt not in by_gt:
            continue
        by_gt[gt][1] += 1
        if r['outcome'] == 'fixed':
            by_gt[gt][0] += 1
    return {gt: (f / t if t else 0.0, t) for gt, (f, t) in by_gt.items()}


def sweep_pool_and_sign(test_subset, alphas=(0.15, 0.3), layer_idx=-10,
                        limit=30, balance_variants=(True, False)):
    results = {}
    for balance in balance_variants:
        for pool_mode in ['mean', 'grad_weighted', 'grad_dir', 'last_token']:
            ts_base = compute_task_steering_vectors(
                paired_dataset, features,
                pool_mode=pool_mode, balance_by_gt=balance,
                normalize=True, verbose=False,
            )
            for sign in [+1, -1]:
                ts = ts_base if sign == +1 else {
                    k: {**v, 'vector': -v['vector']} for k, v in ts_base.items()
                }
                for a in alphas:
                    bal_tag = 'bal' if balance else 'raw'
                    tag = f"{bal_tag}|{pool_mode:14s}|sign={sign:+d}|alpha={a}"
                    print(f"\n>>> {tag}")
                    stats, recs = evaluate_hallucinated(
                        test_subset, model_bundle, task_steering=ts,
                        alpha=a, layer_idx=layer_idx,
                        only_new_tokens=False, alpha_relative=True,
                        prompt_text=PROMPT_TEXT, max_new_tokens=48, limit=limit,
                    )
                    pol = _polarity_breakdown(recs)
                    results[tag] = {
                        'fixed_rate': stats['fixed'] / max(stats['total'], 1),
                        'fixed_gt0': pol[0][0], 'n_gt0': pol[0][1],
                        'fixed_gt1': pol[1][0], 'n_gt1': pol[1][1],
                    }
    print("\n=== ranked (healthy winner: BOTH gt=0 and gt=1 fixed rates > 0) ===")
    rows = sorted(results.items(), key=lambda x: -x[1]['fixed_rate'])
    for tag, r in rows:
        healthy = '*' if (r['fixed_gt0'] > 0 and r['fixed_gt1'] > 0) else ' '
        print(f" {healthy} {tag} | fixed={r['fixed_rate']:.1%} "
              f"| gt=0 {r['fixed_gt0']:.1%} (n={r['n_gt0']}) "
              f"| gt=1 {r['fixed_gt1']:.1%} (n={r['n_gt1']})")
    return results

results = sweep_pool_and_sign(test_dataset, alphas=(0.15, 0.3), layer_idx=-10, limit=10)

In [ ]:
## L=-4 sweep + clean contrast_summarize export
# ------------------------------------------------------------------
# 1. Seed with the L=-3 snapshot results you already have
# ------------------------------------------------------------------
import pandas as pd

contrast_history = [
    # L=-3 snapshot (from your prior run on balanced_test, n=80)
    {'layer': -3, 'alpha': 0.15, 'fixed': 13, 'still': 67, 'unparse': 0,
     'fixed_pct': 16.2, 'gt0_pct': 5.0,  'gt1_pct': 27.5, 'n_gt0': 40, 'n_gt1': 40},
    {'layer': -3, 'alpha': 0.30, 'fixed': 18, 'still': 62, 'unparse': 0,
     'fixed_pct': 22.5, 'gt0_pct': 0.0,  'gt1_pct': 45.0, 'n_gt0': 40, 'n_gt1': 40},
    {'layer': -3, 'alpha': 0.60, 'fixed': 21, 'still': 54, 'unparse': 5,
     'fixed_pct': 26.2, 'gt0_pct': 0.0,  'gt1_pct': 52.5, 'n_gt0': 40, 'n_gt1': 40},
    {'layer': -3, 'alpha': 1.00, 'fixed': 12, 'still': 13, 'unparse': 55,
     'fixed_pct': 15.0, 'gt0_pct': 0.0,  'gt1_pct': 30.0, 'n_gt0': 40, 'n_gt1': 40},
]

# ------------------------------------------------------------------
# 2. Run L=-4  (layer 24 on a 28-layer model)
# ------------------------------------------------------------------
LAYER = -4
for a in [0.15, 0.3, 0.6, 1.0]:
    stats, recs = evaluate_hallucinated(
        balanced_test, model_bundle,
        task_steering=task_steering_ortho_un,
        alpha=a, layer_idx=LAYER,
        prompt_text=PROMPT_TEXT, max_new_tokens=48,
    )
    g = _per_gt(recs)
    n = max(stats['total'], 1)
    contrast_history.append({
        'layer': LAYER, 'alpha': a,
        'fixed': stats['fixed'], 'still': stats['still_hallu'], 'unparse': stats['unparseable'],
        'fixed_pct': round(stats['fixed']/n*100, 1),
        'gt0_pct': round(g[0][0]*100, 1), 'n_gt0': g[0][1],
        'gt1_pct': round(g[1][0]*100, 1), 'n_gt1': g[1][1],
    })

# ------------------------------------------------------------------
# 3. Clean contrast_summarize DataFrame
# ------------------------------------------------------------------
contrast_summarize = pd.DataFrame(contrast_history)
contrast_summarize = contrast_summarize[[
    'layer', 'alpha', 'fixed', 'still', 'unparse',
    'fixed_pct', 'gt0_pct', 'gt1_pct', 'n_gt0', 'n_gt1'
]]

print("\n=== contrast_summarize ===")
print(contrast_summarize.to_string(index=False))

# Export to CSV in the same folder as the notebook
CSV_PATH = 'contrast_summarize.csv'
contrast_summarize.to_csv(CSV_PATH, index=False)
print(f"\nSaved to: {CSV_PATH}")